<a href="https://colab.research.google.com/github/Ayoraham/receipt_scanner_V2/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "Pillow>=10.0.0,<11.0.0" --upgrade
!pip install -qqq torchmetrics
!pip install -qqq pytorch_lightning

import PIL
print(f"Verified Pillow Version: {PIL.__version__}")

In [ ]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForSequenceClassification
from pathlib import Path
import os
from torchmetrics import Accuracy
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split
import json

In [ ]:
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base")
model = LayoutLMv3ForSequenceClassification.from_pretrained('microsoft/layoutlmv3-base')

In [ ]:
!gdown 1bQ4mFbVRUtOEJSe8b4hUYIcngSgfdldw
!tar -xf financial-documents-ocr.tar.xz

In [ ]:
from pathlib import Path
img_pths = sorted(list(Path("images").glob("*/*.jpg")))
print(len(img_pths))

In [ ]:
classes = sorted([p.name for p in Path("images").glob("*")])
classes

In [ ]:
label_2_id = {item:id for id,item in enumerate(classes)}
id_2_label = {val:key for key,val in label_2_id.items()}
id_2_label

#Dataset

In [ ]:
import torch
import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

In [ ]:
def scale_bbox(bbox:list[int],width_scale:[float],height_scale:[float]) -> list[int]:
  return[
      int(bbox[0]*width_scale),
      int(bbox[1]*height_scale),
      int(bbox[2]*width_scale),
      int(bbox[3]*height_scale)
  ]

In [ ]:
processor.image_processor.apply_ocr = False

In [ ]:
from torch.utils.data import Dataset,DataLoader
from PIL import Image
import torch

# The dataset class is what will be passed into the model for training, its also for ease, cuz its wraps your whole dataset in a pytorch dataset class

class DocunmentClassificationDataset(Dataset):

  def __init__(self,image_paths,processor):
    self.image_paths = image_paths
    self.processor = processor

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self,item):
    image_path = self.image_paths[item]
    image = Image.open(image_path).convert("RGB")
    width,height = image.size

    json_path = image_path.with_suffix(".json")

    with json_path.open("r") as f:
      ocr_result = json.load(f)

    width_scale,height_scale = 1000/width, 1000/height

    words = []
    boxes = []
    for row in ocr_result:
      words.append(row['word'])
      boxes.append(scale_bbox(row['bounding_box'],width_scale,height_scale))
    label = classes.index(image_path.parent.name)

    encoding = processor(
        image,
        words,
        boxes=boxes,
        max_length=512,
        truncation=True,
        return_tensors='pt',
        padding="max_length"
    )
    encoding = {k: v.squeeze() for k,v in encoding.items()}
    encoding['labels'] = torch.tensor(label)

    return encoding


In [ ]:
train_images, test_images = train_test_split(img_pths, test_size=0.2)
len(train_images),len(test_images)

In [ ]:
train_dataset = DocunmentClassificationDataset(train_images,processor)
test_dataset = DocunmentClassificationDataset(test_images,processor)

In [ ]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2
)

In [ ]:
import pytorch_lightning as pl
from torchmetrics.classification import MulticlassF1Score
class ModelModule(pl.LightningModule):

  def __init__(self, classes: list):
    super().__init__()
    n_classes = len(classes)
    self.model = LayoutLMv3ForSequenceClassification.from_pretrained(
        "microsoft/layoutlmv3-base",
        num_labels=n_classes
    )
    self.label2id = {item:id for id,item in enumerate(classes)}
    self.id2label = {id:item for id,item in enumerate(classes)}
    self.model.config.id2label = self.id2label
    self.model.config.label2id = self.label2id
    self.train_accuracy = Accuracy(task="multiclass",num_classes=n_classes)
    self.val_accuracy = Accuracy(task="multiclass",num_classes=n_classes)
    self.train_f1 = MulticlassF1Score(num_classes=n_classes, average="macro")
    self.val_f1 = MulticlassF1Score(num_classes=n_classes, average="macro")

    self.model.train()



  def forward(self, input_ids,attention_mask, bbox, pixel_values, labels=None):
    return self.model(
        input_ids,
        attention_mask=attention_mask,
        bbox=bbox,
        pixel_values=pixel_values,
        labels=labels
    )

  def training_step(self,batch,batch_idx):
    labels = batch['labels']
    outputs = self(
        batch["input_ids"],
        batch['attention_mask'],
        batch['bbox'],
        batch['pixel_values'],
        labels
    )
    loss = outputs.loss
    self.log("train_loss",loss)
    preds = torch.argmax(outputs.logits, dim=1)
    self.train_f1(preds,labels)
    self.log("train_f1",self.train_f1,on_step=False,on_epoch=True,prog_bar=True)
    self.train_accuracy(outputs.logits,labels)
    self.log("train_acc",self.train_accuracy, on_step=True, on_epoch=True)
    return loss

  def validation_step(self,batch,batch_idx):
    labels = batch['labels']
    outputs = self(
        batch["input_ids"],
        batch['attention_mask'],
        batch['bbox'],
        batch['pixel_values'],
        labels
    )
    loss = outputs.loss
    self.log("val_loss", loss, prog_bar=True)
    preds = torch.argmax(outputs.logits,dim=1)
    self.val_f1(preds,labels)
    self.log("val_f1", self.val_f1, on_step=False, on_epoch=True)
    self.val_accuracy(outputs.logits,labels)
    self.log("val_acc",self.val_accuracy, on_step=False, on_epoch=True)
    return loss

  def configure_optimizers(self):
    return torch.optim.Adam(self.model.parameters(), lr=0.00001)

In [ ]:
model_module = ModelModule(classes)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

In [ ]:
model_checkpoint = ModelCheckpoint(
    filename= "{epoch}-{step}-{val_loss:.4f}",
    save_last= True,
    save_top_k=3,
    monitor="val_loss",
    mode="min"
)

trainer = pl.Trainer(
    accelerator="gpu",
    precision=16,
    devices=1,
    max_epochs=4,
    callbacks=[
        model_checkpoint
    ],
)

In [ ]:
trainer.fit(model_module, train_dataloader, test_dataloader)

In [ ]:
print(f"Best model path: {model_checkpoint.best_model_path}")

# Loading Model and Processor to HF

In [ ]:
!pip install huggingface_hub

from huggingface_hub import notebook_login


In [ ]:
notebook_login()

In [ ]:
model_module.model.push_to_hub("Ayoraham/layoutlmv3-tutorial")
processor.push_to_hub("Ayoraham/layoutlmv3-tutorial")

# Inference

In [ ]:
processor = LayoutLMv3Processor.from_pretrained("Ayoraham/layoutlmv3-tutorial")
model = LayoutLMv3ForSequenceClassification.from_pretrained("Ayoraham/layoutlmv3-tutorial")

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
model.to(device)

In [ ]:
from PIL import Image

In [ ]:
from PIL import Image
from pathlib import Path
import json
import torch

def get_inference(model, processor, img_pth, classes, device):
    img_pth = Path(img_pth)
    image = Image.open(img_pth).convert("RGB")

    # FIX: Define width and height from the image object
    width, height = image.size

    json_path = img_pth.with_suffix(".json")
    with json_path.open("r") as f:
        ocr_result = json.load(f)

    width_scale, height_scale = 1000/width, 1000/height

    words = []
    boxes = []
    for row in ocr_result:
        words.append(row['word'])
        # Ensure scale_bbox is defined in your notebook!
        boxes.append(scale_bbox(row['bounding_box'], width_scale, height_scale))

    # FIX: Use img_pth (your argument name)
    label = classes.index(img_pth.parent.name)

    encoding = processor(
        image,
        words,
        boxes=boxes,
        max_length=512,
        truncation=True,
        return_tensors='pt',
        padding="max_length"
    )

    # Move to device and ensure Batch dimension exists for the model
    # Note: I removed .squeeze() because the model expects
    encoding = {k: v.to(device) for k, v in encoding.items()}
    encoding['labels'] = torch.tensor([label]).to(device) # Wrapped in list for batching

    with torch.inference_mode():
        output = model(**encoding)
        probs = torch.softmax(output.logits, dim=1)

        # .item() extracts the number from the tensor
        pred_idx = torch.argmax(output.logits, dim=1).item()
        pred_conf = torch.max(probs, dim=1).values.item()

        # FIX: Use brackets [] for dictionary lookup
        pred_label = model.config.id2label[pred_idx]

    return [output, probs, pred_idx, pred_conf, pred_label]

In [ ]:
test = img_pths[3]
test

In [ ]:
x = get_inference(model, processor, test, classes=classes, device=device)

In [ ]:
x